# Train Colosseum on your own dataset

Run this notebook in [Google Colab](https://colab.research.google.com/github/Aqshalikhsan/Colosseum-Microcontroller/blob/main/notebooks/Train_Colosseum.ipynb) or Jupyter with Python 3.11+. In Colab, select a GPU runtime first. Colosseum trains one target class per checkpoint from scratch; the included UC3M weights are not used here.


## 1. Install the framework

The PyPI distribution is `colosseum-microcontroller`; the Python import is `colosseum`.


In [ ]:
%pip install -q --upgrade colosseum-microcontroller
import torch
print('PyTorch:', torch.__version__, '| CUDA available:', torch.cuda.is_available())


## 2. Provide a dataset

Use images and normalized box annotations in this layout:

```text
my_dataset/
  data.yaml
  images/train/...jpg
  images/val/...jpg
  labels/train/...txt
  labels/val/...txt
```

A label line is `class_id center_x center_y width height`, with coordinates from 0 to 1. An image without objects can have an empty or missing label file. This is an annotation format; training uses the Colosseum model and code. Example `data.yaml`:

```yaml
path: .
train: images/train
val: images/val
names: [object]
```

If the dataset already exists in the notebook's working directory, set `DATA_YAML` below. In Colab, if that file is missing, this cell prompts for a ZIP containing exactly one `data.yaml`.


In [ ]:
from pathlib import Path
from zipfile import ZipFile

DATA_YAML = Path('my_dataset/data.yaml')  # Change this if your dataset is already available.
if not DATA_YAML.is_file():
    try:
        from google.colab import files
    except ImportError as exc:
        raise FileNotFoundError('Set DATA_YAML to your existing data.yaml') from exc
    uploaded = files.upload()
    archives = [Path(name) for name in uploaded if name.lower().endswith('.zip')]
    if len(archives) != 1:
        raise ValueError('Upload one ZIP containing the dataset and data.yaml')
    destination = Path('colosseum_data').resolve()
    destination.mkdir(parents=True, exist_ok=True)
    with ZipFile(archives[0]) as archive:
        for member in archive.infolist():
            target = (destination / member.filename).resolve()
            if not target.is_relative_to(destination):
                raise ValueError(f'Unsafe ZIP member: {member.filename}')
        archive.extractall(destination)
    candidates = list(destination.rglob('data.yaml'))
    if len(candidates) != 1:
        raise ValueError('The ZIP must contain exactly one data.yaml')
    DATA_YAML = candidates[0]
DATA_YAML = DATA_YAML.resolve()
print('Dataset config:', DATA_YAML)


## 3. Train your model

Set `LABEL` to the object you want to detect. Use `GRID = '2x8'` for wide, horizontal objects such as license plates; use `4x4` for more balanced shapes. `SOURCE_CLASS` selects one class ID from the annotation files. The default 300 + 20 epochs are the full two-stage recipe; lower them only for a quick pipeline check. Training results are written under `runs/custom/RUN_NAME/seedN/`.


In [ ]:
RUN_NAME = 'my_object'
LABEL = 'my object'
GRID = '4x4'
SOURCE_CLASS = 0
SEEDS = (0,)  # Use (0, 1, 2) for a three-seed experiment.
BASE_EPOCHS = 300
SCORE_EPOCHS = 20
BATCH_SIZE = 16

from colosseum import Colosseum
model = Colosseum()
checkpoints = model.train(
    data=DATA_YAML, name=RUN_NAME, label=LABEL, grid=GRID,
    source_class=SOURCE_CLASS, seeds=SEEDS,
    base_epochs=BASE_EPOCHS, score_epochs=SCORE_EPOCHS,
    batch_size=BATCH_SIZE,
)
print('Checkpoints:', *checkpoints, sep='\n- ')


## 4. Read validation metrics

Each seed writes `metrics.json` beside its checkpoint. AP50, AP75 and AP50:95 are computed on the validation split. Precision, recall and F1 use confidence 0.35 and IoU 0.50.


In [ ]:
import json

metric_keys = ('ap50', 'ap75', 'ap50_95', 'precision', 'recall', 'f1')
for checkpoint in checkpoints:
    report = json.loads(checkpoint.with_name('metrics.json').read_text())
    print(f'Seed {report["seed"]} | ' + ' | '.join(
        f'{key}={report[key]:.4f}' for key in metric_keys
    ))
print('First seed metrics are also available as model.metrics')


## 5. Try prediction on one validation image

This is a visual check on an existing validation image. The saved checkpoint can later be loaded with `Colosseum(checkpoints[0])`.


In [ ]:
import yaml
from PIL import Image, ImageDraw, ImageOps
from IPython.display import display

config = yaml.safe_load(DATA_YAML.read_text())
base = Path(config.get('path', DATA_YAML.parent))
if not base.is_absolute():
    base = (DATA_YAML.parent / base).resolve()
val_images = base / config['val']
image_paths = sorted(p for p in val_images.rglob('*') if p.suffix.lower() in
                     {'.jpg', '.jpeg', '.png', '.bmp', '.webp'})
if not image_paths:
    raise FileNotFoundError(f'No validation images found in {val_images}')
image_path = image_paths[0]
boxes = model.predict(image_path)
with Image.open(image_path) as opened:
    preview = ImageOps.exif_transpose(opened).convert('RGB')
draw = ImageDraw.Draw(preview)
for box in boxes:
    draw.rectangle((box.x1, box.y1, box.x2, box.y2), outline='red', width=3)
    draw.text((box.x1, max(0, box.y1 - 12)),
              f'{box.label} {box.confidence:.2f}', fill='red')
print(image_path, '| detections:', len(boxes))
display(preview)


## 6. Save the results

The ZIP contains each compact checkpoint and its validation metrics. In Colab, uncomment the last two lines to download it.


In [ ]:
from zipfile import ZIP_DEFLATED

result_zip = Path(f'{RUN_NAME}_colosseum_results.zip')
with ZipFile(result_zip, 'w', compression=ZIP_DEFLATED) as archive:
    for checkpoint in checkpoints:
        for result in (checkpoint, checkpoint.with_name('metrics.json')):
            archive.write(result, arcname=f'{checkpoint.parent.name}/{result.name}')
print('Results:', result_zip.resolve())
# from google.colab import files
# files.download(str(result_zip))
